In [10]:
# Негізгі кітапханаларды импорттаймыз
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import time
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # нәтижелер қайталануы үшін фикс random_state


## Датасетті жүктеу және бөлу

**Неге train/test бөлеміз?**  
- Train — модельді үйрету үшін.
- Test — модельдің жаңа деректерде (көрмеген деректерде) қалай жұмыс істейтінін бағалау үшін.

`Stratify=y` қолдану арқылы train/test ішіндегі класстар үлесі шамамен бірдей болып сақталады.


In [2]:
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Classes distribution (train):", np.bincount(y_train))
print("Classes distribution (test):", np.bincount(y_test))


Train size: (426, 30) Test size: (143, 30)
Classes distribution (train): [159 267]
Classes distribution (test): [53 90]


---
#  1-деңгей. Түсінуді тексеру (теория)

## Есеп 1. Анықтамалар


### 1) Ансамбль әдісі деген не?
Ансамбль әдісі — **бірнеше модельді біріктіріп**, олардың болжамдарын *бір ортақ шешімге* келтіретін тәсіл.  
Негізгі идея: әр модельдің қателігі бірдей емес; дұрыс комбинация жалпы қателікті азайтады.

### 2) Неліктен бірнеше модель бір модельден жақсы жұмыс істеуі мүмкін?
Себебі ансамбль:
- **Variance**-ті азайтады (әсіресе bagging типі) — модель "шулы" дерекке тым бейімделмейді.
- Бір модель жіберген қатені басқа модель **компенсация** жасай алады.
- Түрлі random фактор (bootstrap, feature subsampling) модельдерді **диверсификация** жасайды, бұл жалпы нәтижені жақсартады.

### 3) Bias пен Variance деген не?
- **Bias (ығысу)** — модельдің шынайы заңдылықты жеткілікті үйренбеуі (өте қарапайым модель → underfitting).
- **Variance (дисперсия)** — модельдің деректердегі кездейсоқ шумға қатты бейімделуі (өте күрделі модель → overfitting).

Классикалық trade-off: **Bias төмен болса, Variance жоғары болуы мүмкін**, және керісінше.


## Есеп 2. Bagging идеясы


### 1) Bootstrap sample деген не?
**Bootstrap sample** — бастапқы train деректерінен **қайта таңдау (replacement) арқылы** алынған үлгі.  
Яғни бір объект бірнеше рет түсуі мүмкін, ал кейбірі мүлде түспеуі мүмкін.

### 2) Bagging қандай проблеманы азайтады?
Bagging көбіне **Variance-ті азайтады**.  
Мысалы, Decision Tree — variance-і жоғары модель (train-ге тез бейімделеді). Bagging көптеген ағаштың орташа дауысын алып, ауытқуды төмендетеді.

### 3) Bagging параллель орындала ма? Неге?
Иә, **параллель** орындалады, себебі әр base model (ағаш) **бір-бірінен тәуелсіз** bootstrap үлгілерде үйретіледі.  
Яғни sequential тәуелділік жоқ → parallel training мүмкін.


## Есеп 3. Random Forest ерекшелігі

Төмендегілердің қайсысы Random Forest-ке тән?


-  **Bootstrap sampling** — Иә (әр ағаш bootstrap үлгіде үйренеді).  
-  **Feature randomness** — Иә (әр split кезінде feature-тердің кездейсоқ subset-і қаралады).  
-  **Sequential learning** — Жоқ (бұл boosting-ке тән).  
-  **Decision trees** — Иә (base learner ретінде ағаштар).  
-  **Gradient optimization** — Жоқ (бұл gradient boosting логикасына тән).


---
#  2-деңгей. Есептеу және логика

## Есеп 4. Majority voting

3 классификатор:
- M1 → 1
- M2 → 0
- M3 → 1


### 1) Ансамбль болжамы қандай?
**Majority vote**: (1, 0, 1) → екі "1" бар → **ансамбль нәтижесі = 1**.

### 2) Егер бір модельдің салмағы 2 есе үлкен болса, нәтиже өзгере ме?
Салмақты дауыс беруде (weighted voting) нәтиже салмақтарға тәуелді.

Мысал:
- Егер 0 берген модельдің салмағы 2 болса:  
  1 (w=1) + 0 (w=2) + 1 (w=1) → "0" жинағы 2, "1" жинағы 2 → тең.  
  Тең жағдайда көбіне ереже керек (мысалы, 1-ді таңдау немесе class prior).  
- Егер 1 берген модельдердің біреуіне салмақ 2 болса:  
  "1" жинағы 3 болады → нәтиже **1** болып қалады.

Қорытынды: **иә, өзгеруі мүмкін**, нақты қай модельге салмақ берілгеніне байланысты.


---
#  3-деңгей. Bagging & Random Forest (практика)

## Есеп 6. Bagging (Python)
Тапсырма:
- Base model: Decision Tree
- n_estimators = 30
- Accuracy есептеу
- Бір ағашпен салыстыру


In [3]:
# 1) Бір Decision Tree (baseline) модель
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)

print("Single Decision Tree accuracy:", dt_acc)


Single Decision Tree accuracy: 0.9230769230769231


In [4]:
# 2) Bagging (Decision Tree base learner)
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    n_estimators=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
bag.fit(X_train, y_train)
bag_pred = bag.predict(X_test)
bag_acc = accuracy_score(y_test, bag_pred)

print("Bagging (30 trees) accuracy:", bag_acc)
print("\nΔ improvement:", bag_acc - dt_acc)


Bagging (30 trees) accuracy: 0.958041958041958

Δ improvement: 0.03496503496503489


### Нәтижені түсіндіру
- Decision Tree жалғыз өзі variance-і жоғары болуы мүмкін.
- Bagging көптеген ағаштың ортақ шешімін алады → variance азаяды → тесттегі accuracy жиі өседі.


## Есеп 7. Random Forest гиперпараметрлері


### 1) n_estimators артса не болады?
- Орташа есеппен модель **тұрақтырақ** болады (variance азаяды).
- Бірақ есептеу уақыты мен RAM қолдану өседі.
- Белгілі бір нүктеден кейін accuracy өсімі азаяды (diminishing returns).

### 2) max_depth азайса әсері?
- Ағаштар қарапайым болады → **Bias өседі**, **Variance азаяды**.
- Overfitting тәуекелі төмендейді, бірақ underfitting болуы мүмкін.

### 3) Неліктен Random Forest overfitting-ке төзімді?
- Bootstrap sampling + feature randomness → ағаштар **бір-біріне ұқсамайды** (decorrelation).
- Орташа дауыс/орташа ықтималдық → бір ағаштың артық бейімделуі жалпы нәтижеге аз әсер етеді.


Random Forest-ті де есептеп көрейік (қысқа тәжірибе).


In [5]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print("Random Forest accuracy:", rf_acc)


Random Forest accuracy: 0.958041958041958


---
#  4-деңгей. Gradient Boosting

## Есеп 8. Бустинг логикасы


### 1) Неліктен Gradient Boosting sequential?
Boosting-те әр жаңа модель **алдыңғы модельдердің қателігін (residual) түзету** үшін үйретіледі.  
Сондықтан `model_{t+1}` үйренуі `model_t` нәтижесіне тәуелді → **sequential**.

### 2) Қалдықтар (residuals) деген не?
Қалдық (residual) — "нақты мән" мен "модель болжамы" арасындағы айырма.  
Идея: келесі модель осы айырманы азайтуға тырысады.

### 3) Learning rate-тің рөлі қандай?
Learning rate — әр жаңа "әлсіз модельдің" жалпы ансамбльге қосатын үлесін **бәсеңдетеді**.
- Кішкентай learning rate → баяу, бірақ жиі жақсы generalization
- Үлкен learning rate → тез, бірақ overfitting қаупі жоғары


## Есеп 9. Gradient Boosting (Python)
Тапсырма:
- GradientBoostingClassifier
- n_estimators = 50, 100, 200
- Accuracy салыстыру, қорытынды


In [6]:
def eval_gb(n_estimators):
    gb = GradientBoostingClassifier(
        n_estimators=n_estimators,
        random_state=RANDOM_STATE
    )
    gb.fit(X_train, y_train)
    pred = gb.predict(X_test)
    acc = accuracy_score(y_test, pred)
    return acc

gb_results = {n: eval_gb(n) for n in [50, 100, 200]}
gb_results


{50: 0.9440559440559441, 100: 0.958041958041958, 200: 0.958041958041958}

In [7]:
for n, acc in gb_results.items():
    print(f"GradientBoosting n_estimators={n:3d} -> accuracy={acc:.4f}")


GradientBoosting n_estimators= 50 -> accuracy=0.9441
GradientBoosting n_estimators=100 -> accuracy=0.9580
GradientBoosting n_estimators=200 -> accuracy=0.9580


### Қорытынды (жалпы логика)
- n_estimators артқанда модель қуаттырақ болады, бірақ overfitting қаупі де өседі.
- Практикада ең жақсы мән — дерекке, learning rate-ке және regularization-ға тәуелді.


---
#  5-деңгей. Stacking

## Есеп 10. Stacking құрылымы
Берілген модельдер:
- Decision Tree
- Random Forest
- SVM


### 1) Қайсысы Level-0 модель?
**Level-0** — базалық модельдер (base learners). Берілген үшеуі де Level-0 бола алады.

### 2) Мета-модель қандай болуы мүмкін?
Көбіне қарапайым және тұрақты модель:
- Logistic Regression (жиі)
- Linear model
- Light regularized model

### 3) Неліктен train деректерді тікелей қолдануға болмайды?
Егер meta-model-ге базалық модельдердің train-дегі predictions берсек:
- Базалық модель train-де өте жақсы (тіпті overfit) болуы мүмкін
- Meta-model "жалған" өте оптимистік predictions-ті көріп, **data leakage** болады
- Нәтиже test-де құлайды

Сондықтан stacking үшін дұрыс тәсіл — **out-of-fold predictions** (CV арқылы) қолдану.


## Есеп 11. Stacking (Python)
Тапсырма:
- Кемінде 2 базалық модель
- Logistic Regression — meta-model
- Accuracy есептеу
- Random Forest-пен салыстыру

Төменде `StackingClassifier` қолданамыз. SVM үшін scaling маңызды, сондықтан Pipeline жасаймыз.


In [8]:
from sklearn.ensemble import StackingClassifier

# Level-0 (base) модельдер
estimators = [
    ("dt", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ("svm", Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
    ])),
]

# Meta-model (Level-1)
meta = LogisticRegression(max_iter=500, random_state=RANDOM_STATE)

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=meta,
    stack_method="predict_proba",  # классификацияда көбіне proba тиімді
    cv=5,
    n_jobs=-1
)

stack.fit(X_train, y_train)
stack_pred = stack.predict(X_test)
stack_acc = accuracy_score(y_test, stack_pred)

print("Stacking accuracy:", stack_acc)
print("Random Forest accuracy:", rf_acc)
print("Δ stacking - rf:", stack_acc - rf_acc)


Stacking accuracy: 0.965034965034965
Random Forest accuracy: 0.958041958041958
Δ stacking - rf: 0.006993006993006978


---
#  6-деңгей. Аналитикалық

## Есеп 12. Bias–Variance талдауы


Төменде салыстыру — жалпы тенденция (әр деректе шамалы өзгеруі мүмкін):

### 1) Single Decision Tree
- Bias: төмен/орташа (терең болса)
- Variance: жоғары (дерек өзгерсе, ағаш қатты өзгереді)

### 2) Bagging
- Bias: шамамен сол деңгей (ағаш bias-ы қатты өзгермейді)
- Variance: **төмендейді** (орташа алу арқылы)

### 3) Random Forest
- Bias: bagging-ке жақын немесе сәл жоғары (feature randomness әсері)
- Variance: **тағы да төмен** (ағаштардың корреляциясы азаяды)

### 4) Gradient Boosting
- Bias: **төмендейді** (қателікті біртіндеп түзетеді)
- Variance: өсіп кетуі мүмкін (әсіресе көп итерация, үлкен learning rate) → overfitting қаупі бар


## Есеп 13. Қай әдісті таңдау керек?


### 1) Шағын датасет
- **Bagging / Random Forest** — тұрақтылық үшін жақсы.
- Gradient Boosting та жақсы болуы мүмкін, бірақ hyperparameter tuning қажет.

### 2) Үлкен noisy деректер
- **Random Forest** (noisy-ге төзімді, variance control)
- Bagging та жарайды.

### 3) Feature саны өте көп
- Random Forest немесе boosting (кейде regularized linear/SVM де).
- Егер sparse және өте үлкен болса, басқа әдістер де тиімді (мысалы, linear models), бірақ ансамбль контекстінде: **Random Forest** жиі жақсы baseline.

### 4) Ең жоғары accuracy қажет
- Көбіне **Gradient Boosting** (жақсы тюнингпен) немесе stacking.
- Бірақ уақыт/ресурс/түсіндірмелілік талаптарын ескеру керек.


---
#  Қорытынды есеп (Mini-project) — Есеп 14. Толық салыстыру

Бір датасетте:
- Bagging
- Random Forest
- Gradient Boosting
- Stacking

Тапсырма:
- Барлығын үйрету
- Accuracy салыстыру
- Қайсысы ең тұрақты?
- Қайсысы ең баяу?

Төменде:
1) Test accuracy (бір split)
2) 5-fold Cross-Validation mean ± std (тұрақтылық үшін)
3) Training time (шамамен)


In [9]:
# Utility: модельді өлшеу функциясы
def benchmark_model(name, model):
    # Уақыт өлшеу (train + predict)
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred = model.predict(X_test)
    pred_time = time.perf_counter() - t1

    acc_test = accuracy_score(y_test, pred)

    # CV: тұрақтылық (mean және std)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

    return {
        "model": name,
        "test_accuracy": acc_test,
        "cv_mean": float(np.mean(cv_scores)),
        "cv_std": float(np.std(cv_scores)),
        "train_time_sec": train_time,
        "predict_time_sec": pred_time
    }

models = []

# Bagging (30 trees)
models.append(("Bagging", BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    n_estimators=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)))

# Random Forest
models.append(("RandomForest", RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)))

# Gradient Boosting
models.append(("GradientBoosting", GradientBoostingClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE
)))

# Stacking (dt + rf + svm) -> logistic regression meta
stack_full = StackingClassifier(
    estimators=[
        ("dt", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ("rf", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
        ("svm", Pipeline([("scaler", StandardScaler()),
                         ("svc", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))])),
    ],
    final_estimator=LogisticRegression(max_iter=800, random_state=RANDOM_STATE),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)
models.append(("Stacking", stack_full))

results = []
for name, model in models:
    results.append(benchmark_model(name, model))

df_results = pd.DataFrame(results).sort_values(by="cv_mean", ascending=False)
df_results


,model,test_accuracy,cv_mean,cv_std,train_time_sec,predict_time_sec
3,Stacking,0.979021,0.968374,0.013110,5.317533,0.050257
2,GradientBoosting,0.958042,0.957864,0.024405,1.537335,0.001346
0,Bagging,0.958042,0.954339,0.021730,0.235039,0.022621
1,RandomForest,0.958042,0.952569,0.013071,0.885905,0.078195


In [ ]:
# Нәтижені әдемі шығару
pd.set_option("display.precision", 4)
df_results.reset_index(drop=True)


### Mini-project қорытындысын қалай оқу керек?

- **Ең тұрақты**: CV standard deviation (`cv_std`) ең кішісі — модель әр түрлі fold-та бірдей нәтиже береді.
- **Ең баяу**: `train_time_sec` (және кейде predict_time) ең үлкені.
- Accuracy үшін `cv_mean` (жалпы орташа) және `test_accuracy` (бір split) қатар қаралады.


---
# Қосымша теория сұрақтары

### 1) Неліктен Random Forest feature importance бере алады?
Random Forest — шешім ағаштар жиыны. Ағаштарда split жасағанда feature-тер ақпараттық ұтыс (impurity decrease) береді.  
Орташа есеппен барлық ағаштағы осы "үлестерді" жинақтау арқылы feature importance есептеледі (мысалы, Gini importance).

### 2) Gradient Boosting неге overfitting жасай алады?
- Sequential түрде қателікті үздіксіз азайтады → train-ге тым жақсы бейімделуі мүмкін.
- Үлкен `n_estimators`, үлкен `learning_rate`, немесе терең base learners → overfit тәуекелі артады.

### 3) Stacking қашан тиімді емес?
- Датасет өте кішкентай болса (CV predictions noisy болады)
- Base модельдер бір-біріне тым ұқсас болса (диверсификация жоқ)
- Уақыт/ресурс шектеулі болса (stacking есептеу жағынан ауыр)

### 4) Bagging bias-ты азайта ма?
Көбіне **жоқ** (немесе өте аз). Bagging негізгі әсері — **variance-ті азайту**.  
Bias-ты азайту көбіне boosting-ке тән.


---
# Қорытынды
Осы ноутбук арқылы сіз:
- Ансамбль әдістерінің негізгі теориясын түсіндірдіңіз
- Bagging, Random Forest, Gradient Boosting, Stacking модельдерін Python-да үйретіп салыстырдыңыз
- Bias–Variance тұрғысынан талдау жасадыңыз
- Бір датасетте толық салыстыру mini-project орындадыңыз

Егер мұғалім/оқытушы қосымша талап қойса (мысалы, ROC-AUC, F1-score, feature importance графигі), осы ноутбукке оңай қосып беремін.
